# Example of running power flows for a scenario
This requires the save file from running 'scenario_example.ipynb' which has a network model and results from energy optimisation. This file will be first loaded to give our scenario


In [1]:
import sys
sys.path.append("../") # go to parent dir
import echo_scenario as ecs

file_name = '../data/test_scenario.pickle'

scenario = ecs.EchoScenario()
scenario.load(file_name)

Importing network data
Finished importing network data


# Run the power flows
Next we will run the power flows, current a single power factor can be set for all sites. A save file name can also be specified to save the results automatically. This should be a '.pickle' file. Additionally, the file will be compressed using lzma.

In [2]:
power_factor = 0.93
save_file_name = '../data/power_flow_results.pickle'
power_flow_results = scenario.run_power_flows(power_factor=power_factor, save_pickle_file=save_file_name)

Running power flows: 100%|██████████| 672/672 [02:49<00:00,  3.96it/s]


# Having a look at some of the results
The results are output as a dictionary of pandas dataframes. The data frames are:
- status - list of true false for whether the power flow converged
- bus_voltage
- branch_power_0: power at end 0 of branch
- branch_power_1: power at end 1 of branch
- branch_current_0: current at end 0 of branch
- branch_current_1: current at end 1 of branch
- transformer_names: a list of the names of all transformers, note that within the dataframes these names would be appended with the phase (_A, _B, _C)

Branches include lines and transformers. To get just the transformers look for the columns that contain the transformer names.

Positive power at an end represents power injected into the branch, and negative represents power leaving the branch. For example, to work out power losses in a branch you would sum the power at end 0 and end 1.

In [9]:
power_flow_results.keys()

dict_keys(['status', 'bus_voltage', 'branch_power_0', 'branch_power_1', 'branch_current_0', 'branch_current_1', 'transformer_names'])

# Check if the power flow calculations converged

In [12]:
import numpy as np
print('Number of timesteps for which the power flow converged was {}'.format(np.array(power_flow_results['status']).sum()))
print('Number of timesteps for which the power flow did not converged was {}'.format(len(power_flow_results['status']) - np.array(power_flow_results['status']).sum()))

Number of timesteps for which the power flow converged was 672
Number of timesteps for which the power flow did not converged was 0


In [5]:
power_flow_results['bus_voltage'].head()

,node_1.A,node_1.B,node_1.C,node_10.A,node_10.B,node_10.C,node_100.A,node_100.B,node_100.C,node_1000.A,...,node_994.C,node_995.A,node_995.B,node_995.C,node_996.B,node_997.A,node_998.A,node_998.B,node_998.C,node_999.A
0,0.995455,0.995455,0.995455,0.914483,0.916030,0.959051,0.909224,0.923083,0.954388,0.945752,...,0.980015,0.948646,0.965337,0.982753,0.964269,0.947163,0.947042,0.967186,0.983502,0.946060
1,0.995455,0.995455,0.995455,0.907635,0.919157,0.959015,0.902389,0.926297,0.954377,0.946843,...,0.980439,0.949721,0.966164,0.983137,0.965118,0.948267,0.948107,0.967959,0.983895,0.947145
2,0.995455,0.995455,0.995455,0.901362,0.913818,0.955548,0.896030,0.920687,0.950884,0.941909,...,0.978503,0.944868,0.962614,0.981410,0.961458,0.943259,0.943307,0.964694,0.982122,0.942243
3,0.995455,0.995455,0.995455,0.918074,0.941457,0.953706,0.910231,0.948071,0.949850,0.946590,...,0.980232,0.949476,0.965797,0.982950,0.964740,0.948008,0.947867,0.967618,0.983704,0.946895
4,0.995455,0.995455,0.995455,0.892516,0.918888,0.956547,0.883624,0.928719,0.950911,0.944130,...,0.979389,0.947051,0.963784,0.982197,0.962679,0.945516,0.945464,0.965728,0.982931,0.944449


# Get minimum and maximum voltage (PU)

In [8]:
print('max voltage was {} (pu)'.format(power_flow_results['bus_voltage'].max().max()))
print('min voltage was {} (pu)'.format(power_flow_results['bus_voltage'].min().min()))

max voltage was 1.0815685438564857 (pu)
min voltage was 0.6324890842729558 (pu)
